# Phase 7 — Kaggle (Groq) runner

Runs the Groq-backed Phase 7 experiments (synthetic causal, MMR baseline, CRAG baseline) on Kaggle's free CPU runtime with Groq doing all the LLM work via API.

**Wall-clock**: ~30-60 min total depending on selected scripts. **Internet must be ON**, `GROQ_API_KEY` must be in Kaggle Secrets.

If you also set `GH_TOKEN` (Kaggle Secret), the last cell pushes results to a `kaggle-phase7` branch on `Saket-Maganti/rag-hallucination-detection`.

**Daily Groq budget**: 100k tokens/day total per org. Each Phase 7 script uses 30-50k tokens. Plan 1-2 scripts/day if you have other Groq usage.

## Selected scripts

This notebook will run: **synthetic_causal, mmr_baseline, crag_baseline**.


In [ ]:
# 1. Pull the repo (read-only clone is fine)
import os, subprocess, pathlib
REPO_URL = 'https://github.com/Saket-Maganti/rag-hallucination-detection.git'
ROOT = pathlib.Path('/kaggle/working/rag')
if not ROOT.exists():
    subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(ROOT)])
os.chdir(ROOT)
subprocess.check_call(['git', 'log', '--oneline', '-1'])


In [ ]:
# 2. Install lean dependencies (Groq is API-only — no torch GPU)
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'groq', 'sentence-transformers', 'chromadb', 'datasets', 'langchain', 'langchain-community', 'langchain-ollama', 'pandas', 'scipy'])


In [ ]:
# 3. Pull GROQ_API_KEY from Kaggle Secrets
from kaggle_secrets import UserSecretsClient
secrets = UserSecretsClient()
import os
os.environ['GROQ_API_KEY'] = secrets.get_secret('GROQ_API_KEY')
print('GROQ_API_KEY set:', bool(os.environ.get('GROQ_API_KEY')))


In [ ]:
# 4. Smoke test before the long runs
import subprocess, sys
subprocess.check_call([sys.executable, 'scripts/smoke_test_groq.py', '--models', 'llama-3.3-70b'])


In [ ]:
# Run Phase 7: synthetic_causal
import subprocess, sys, time
print('=== synthetic_causal starting ===')
t0 = time.time()
subprocess.check_call([sys.executable, 'experiments/run_synthetic_causal.py', '--backend', 'groq', '--model', 'llama-3.3-70b', '--datasets'] + 'squad pubmedqa'.split() + ['--n', '100'])
print(f'=== {} done in {{:.1f}} min ==='.format('synthetic_causal', (time.time() - t0) / 60))


In [ ]:
# Run Phase 7: mmr_baseline
import subprocess, sys, time
print('=== mmr_baseline starting ===')
t0 = time.time()
subprocess.check_call([sys.executable, 'experiments/run_mmr_baseline.py', '--backend', 'groq', '--model', 'llama-3.3-70b', '--datasets'] + 'squad pubmedqa'.split() + ['--n', '30'])
print(f'=== {} done in {{:.1f}} min ==='.format('mmr_baseline', (time.time() - t0) / 60))


In [ ]:
# Run Phase 7: crag_baseline
import subprocess, sys, time
print('=== crag_baseline starting ===')
t0 = time.time()
subprocess.check_call([sys.executable, 'experiments/run_crag_baseline.py', '--backend', 'groq', '--model', 'llama-3.3-70b', '--datasets'] + 'squad pubmedqa'.split() + ['--n', '30'])
print(f'=== {} done in {{:.1f}} min ==='.format('crag_baseline', (time.time() - t0) / 60))


In [ ]:
# 5. Snapshot results into a downloadable zip + optional GH push
import shutil, subprocess, os, pathlib
OUT = pathlib.Path('/kaggle/working')
# Bundle whatever Phase 7 result dirs exist
phase7_dirs = ['synthetic_causal', 'mmr_baseline', 'crag_baseline',
                'ccs_alternatives', 'failure_typology']
import os
existing = [f'results/{d}' for d in phase7_dirs if os.path.isdir(f'results/{d}')]
if existing:
    subprocess.check_call(['tar', 'czf', str(OUT / 'phase7_results.tar.gz')] + existing)
    print('Wrote', OUT / 'phase7_results.tar.gz')
else:
    print('No Phase 7 result dirs found yet')

# Optional: push to GitHub if GH_TOKEN secret is present.
try:
    from kaggle_secrets import UserSecretsClient
    gh_token = UserSecretsClient().get_secret('GH_TOKEN')
except Exception:
    gh_token = None
if gh_token and existing:
    REPO = 'Saket-Maganti/rag-hallucination-detection'
    branch = 'kaggle-phase7'
    subprocess.check_call(['git', 'config', 'user.email', 'kaggle@auto'])
    subprocess.check_call(['git', 'config', 'user.name', 'kaggle-runner'])
    subprocess.check_call(['git', 'checkout', '-B', branch])
    for d in existing:
        subprocess.check_call(['git', 'add', d])
    subprocess.check_call(['git', 'commit', '-m', 'Phase 7 results from Kaggle/Groq'])
    push_url = f'https://{gh_token}@github.com/{REPO}.git'
    subprocess.check_call(['git', 'push', '-f', push_url, branch])
    print(f'Pushed to {REPO} {branch}')
else:
    if not gh_token:
        print('No GH_TOKEN — download phase7_results.tar.gz manually.')


## Where the results land

After this notebook completes (with `GH_TOKEN`):

  - GitHub branch `kaggle-phase7` on `Saket-Maganti/rag-hallucination-detection` has all CSVs/JSON.
  - Local: `git pull origin kaggle-phase7` then `git merge kaggle-phase7` to bring into main.

Without `GH_TOKEN`:

  - Download `phase7_results.tar.gz` from the Output panel.
  - Extract to your local `results/` and commit normally.
